In [1]:
!pip install fastapi uvicorn python-multipart rasterio scipy plotly pyngrok nest-asyncio

In [ ]:
!pip install pyngrok
import io
import asyncio
import nest_asyncio
import numpy as np
import plotly.graph_objects as go
import rasterio
from scipy.ndimage import gaussian_filter
from fastapi import FastAPI, File, UploadFile, HTTPException
from fastapi.responses import HTMLResponse
from pyngrok import ngrok
import uvicorn
import time
import uuid

# Allow nested event loops in Colab environment
nest_asyncio.apply()

app = FastAPI(
    title="GeoTIFF 3D Terrain Service",
    description="API for rendering 3D topographic meshes and ground-level views from GeoTIFF imagery.",
)

# Shared topographic color palette
TERRAIN_COLORSCALE = [
    [0.0, "rgb(49, 130, 189)"],    # Valley Blue
    [0.3, "rgb(158, 202, 225)"],  # Light Blue
    [0.6, "rgb(230, 245, 201)"],  # Lowland Cream
    [0.85, "rgb(253, 174, 97)"],  # Ridge Tan
    [1.0, "rgb(166, 97, 26)"],     # Peak Brown
]


def process_geotiff(file_bytes: bytes, max_points: int = 120):
    try:
        with rasterio.open(io.BytesIO(file_bytes)) as src:
            if src.count >= 3:
                r, g, b = src.read(1), src.read(2), src.read(3)
            else:
                r = g = b = src.read(1)

            height, width = r.shape
            step_x = max(1, width // max_points)
            step_y = max(1, height // max_points)

            r_sub = r[::step_y, ::step_x].astype(float)
            g_sub = g[::step_y, ::step_x].astype(float)
            b_sub = b[::step_y, ::step_x].astype(float)

        raw_elevation = 0.2989 * r_sub + 0.5870 * g_sub + 0.1140 * b_sub
        smooth_elevation = gaussian_filter(raw_elevation, sigma=2.5)

        z_min, z_max = smooth_elevation.min(), smooth_elevation.max()
        z_normalized = ((smooth_elevation - z_min) / (z_max - z_min + 1e-6)) * 100.0
        z_aligned = np.flipud(z_normalized)

        x_coords = np.arange(z_aligned.shape[1])
        y_coords = np.arange(z_aligned.shape[0])

        return x_coords, y_coords, z_aligned

    except Exception as e:
        raise HTTPException(status_code=400, detail=f"Processing failed: {str(e)}")


def build_grid_style():
    return dict(
        showgrid=True,
        gridcolor="rgba(80, 80, 80, 0.6)",
        gridwidth=2,
        showline=True,
        linecolor="black",
        linewidth=2,
        zeroline=True,
        zerolinecolor="black",
        showbackground=True,
        backgroundcolor="rgba(240, 240, 240, 0.5)",
        visible=True,
    )


@app.get("/")
def root():
    return {"status": "Colab API Active", "docs": "/docs"}


@app.post("/render/3d-mesh", response_class=HTMLResponse)
async def render_3d_mesh(file: UploadFile = File(...), max_points: int = 120):
    contents = await file.read()
    x_coords, y_coords, z_aligned = process_geotiff(contents, max_points)

    fig = go.Figure(
        data=[
            go.Surface(
                x=x_coords,
                y=y_coords,
                z=z_aligned,
                surfacecolor=z_aligned,
                colorscale=TERRAIN_COLORSCALE,
                showscale=False,
                lighting=dict(ambient=0.75, diffuse=0.8, fresnel=0.1, specular=0.1, roughness=0.6),
                hovertemplate="X: %{x}<br>Y: %{y}<br>Height: %{z:.1f}m<extra></extra>",
            )
        ]
    )

    grid_style = build_grid_style()
    fig.update_layout(
        scene=dict(
            xaxis=dict(title="X Axis", **grid_style),
            yaxis=dict(title="Y Axis", **grid_style),
            zaxis=dict(title="Height (m)", **grid_style),
            aspectratio=dict(x=1, y=1, z=0.2),
            camera=dict(eye=dict(x=1.3, y=-1.3, z=0.8)),
        ),
        paper_bgcolor="white",
        plot_bgcolor="white",
        margin=dict(l=10, r=10, b=10, t=10),
    )
    return HTMLResponse(content=fig.to_html(full_html=True, include_plotlyjs="cdn"))


@app.post("/render/ground-view", response_class=HTMLResponse)
async def render_ground_view(file: UploadFile = File(...), max_points: int = 120):
    contents = await file.read()
    x_coords, y_coords, z_aligned = process_geotiff(contents, max_points)

    fig = go.Figure(
        data=[
            go.Surface(
                x=x_coords,
                y=y_coords,
                z=z_aligned,
                surfacecolor=z_aligned,
                colorscale=TERRAIN_COLORSCALE,
                showscale=False,
                lighting=dict(ambient=0.8, diffuse=0.9, fresnel=0.2, specular=0.1, roughness=0.5),
                hovertemplate="X: %{x}<br>Y: %{y}<br>Elevation: %{z:.1f}m<extra></extra>",
            )
        ]
    )

    grid_style = build_grid_style()
    fig.update_layout(
        scene=dict(
            xaxis=dict(title="X (Distance)", **grid_style),
            yaxis=dict(title="Y (Distance)", **grid_style),
            zaxis=dict(title="Elevation (m)", **grid_style),
            aspectratio=dict(x=1, y=1, z=0.25),
            camera=dict(
                projection=dict(type="perspective"),
                eye=dict(x=0.1, y=-0.8, z=0.12),
                center=dict(x=0.0, y=0.2, z=0.05),
                up=dict(x=0, y=0, z=1),
            ),
            bgcolor="rgba(0,0,0,0)",
        ),
        paper_bgcolor="white",
        plot_bgcolor="white",
        margin=dict(l=0, r=0, b=0, t=0),
    )
    return HTMLResponse(content=fig.to_html(full_html=True, include_plotlyjs="cdn"))


# Set ngrok auth token
ngrok.set_auth_token("3JALbLGbUstScoMIi44k4bmzaoi_9SCEKUBS1tg16CJbGDU8")

# Aggressive Cleanup
try:
    tunnels = ngrok.get_tunnels()
    for t in tunnels:
        ngrok.disconnect(t.public_url)
    ngrok.kill()
    print("Tunnels disconnected and process killed.")
except Exception as e:
    print(f"Cleanup status: {e}")

time.sleep(5)

# Attempt connection on port 8001
try:
    unique_name = f"colab-tunnel-{uuid.uuid4().hex[:8]}"
    public_url = ngrok.connect(8001, name=unique_name)
    print(f" Public API URL: {public_url}")
    print(f" Interactive Swagger UI Docs: {public_url}/docs")
except Exception as e:
    print(f"Failed to connect. If ERR_NGROK_334 persists, visit https://dashboard.ngrok.com/tunnels/agents.")
    raise e

# Run FastAPI server on port 8001
config = uvicorn.Config(app, host="127.0.0.1", port=8001, log_level="info")
server = uvicorn.Server(config)
await server.serve()

Tunnels disconnected and process killed.


INFO:     Started server process [2435]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8001 (Press CTRL+C to quit)


 Public API URL: NgrokTunnel: "https://enviable-shown-shawl.ngrok-free.dev" -> "http://localhost:8001"
 Interactive Swagger UI Docs: NgrokTunnel: "https://enviable-shown-shawl.ngrok-free.dev" -> "http://localhost:8001"/docs
